# Week 2 — Town Visualization

**My contribution:** visualize the town that my teammates create. This notebook does **not** build a new town or simulate infections. It uses Teammate 3's actual `town_snapshots` and `town_meetings` from the existing Week 2 assignment.

The display has 100 residents (one circle per person). Colors: green = S, yellow = E, red = I, blue = R. Optional thin gray lines represent meetings.

## Before running this notebook

1. Teammates 2 and 3 complete their work in `assignments/Week_02_Assignment.ipynb`.
2. After they run their cells, they export their results with the helper in `simulations/town_simulation.py`:

```python
from simulations.town_simulation import save_town_results
save_town_results(town_snapshots, town_meetings)
```

If the assignment notebook is running from the `assignments` folder, import from `../simulations` as explained in the helper file. This produces `simulations/town_data.json`.

3. Run this notebook. **Until your teammates export their results, no actual visualization is available.**

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Works when Jupyter is launched from either repository root or visualizations/
possible_files = [
    Path("simulations/town_data.json"),
    Path("../simulations/town_data.json"),
]
data_file = next((p for p in possible_files if p.exists()), None)
if data_file is None:
    raise FileNotFoundError(
        "No teammates' data found. Ask Teammate 3 to export "
        "town_snapshots and town_meetings to simulations/town_data.json."
    )

with data_file.open(encoding="utf-8") as f:
    data = json.load(f)

town_snapshots = data["town_snapshots"]
town_meetings = data["town_meetings"]
assert len(town_snapshots) == len(town_meetings) == 61
assert all(len(states) == 100 for states in town_snapshots)
print("Loaded teammates' actual town data for days 0–60.")

## Draw the town

We arrange the 100 people in a 10 × 10 grid. Each person keeps the same position across days so it is easier to see their changing status. The grid positions are only for visualization; they do not represent actual geographic locations or distances in the simulation.

In [ ]:
state_colors = {"S": "seagreen", "E": "gold", "I": "tomato", "R": "cornflowerblue"}
positions = {person_id: (person_id % 10, 9 - person_id // 10) for person_id in range(100)}

def show_town(day, show_meetings=False):
    if not 0 <= day < len(town_snapshots):
        raise ValueError("Choose a day from 0 to 60.")

    states = town_snapshots[day]
    fig, ax = plt.subplots(figsize=(7, 7))

    if show_meetings:
        for person_a, person_b in town_meetings[day]:
            x1, y1 = positions[person_a]
            x2, y2 = positions[person_b]
            ax.plot([x1, x2], [y1, y2], color="lightgray", alpha=0.35, zorder=1)

    for person_id, state in enumerate(states):
        x, y = positions[person_id]
        ax.scatter(x, y, s=250, color=state_colors[state],
                   edgecolor="white", linewidth=1, zorder=2)

    counts = {state: states.count(state) for state in "SEIR"}
    ax.set_title(
        f"Town on Day {day}  |  S={counts['S']}  E={counts['E']}  "
        f"I={counts['I']}  R={counts['R']}"
    )
    ax.set_xlim(-0.7, 9.7)
    ax.set_ylim(-0.7, 9.7)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    legend = [Line2D([0], [0], marker="o", color="w", label=label,
                     markerfacecolor=state_colors[state], markersize=10)
              for state, label in [("S", "Susceptible"), ("E", "Exposed"),
                                   ("I", "Infectious"), ("R", "Recovered")]]
    ax.legend(handles=legend, loc="upper center",
              bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False)
    plt.show()

## View the town on important days

These figures show your teammates' results for Day 0, 15, 30, 45, and 60. You can also call `show_town(10)` for any other day.

In [ ]:
for day in [0, 15, 30, 45, 60]:
    show_town(day)

## Optional: Show the meetings

Gray lines connect residents who met during the chosen day. Use `show_town(10, show_meetings=True)` after your teammates have recorded the meeting pairs.

In [ ]:
# Uncomment this line after loading your teammates' data:
# show_town(10, show_meetings=True)

## Connection to the team assignment

The visualization **depends on** the 61 individual-state snapshots produced by Teammate 3. The five daily total lists alone are not enough to identify each resident's state, so this notebook intentionally avoids inventing or randomly generating missing data.

The original `assignments/Week_02_Assignment.ipynb` remains unchanged.